# *<b>LABORATORIO 2 </b>*

Nombres<br>
Laura Carolina Avelino Barrera 202412164
<br>
Gabriela Gonzalez Gomez 202121554



### Importación de librerías

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
import matplotlib.pyplot as plt


print("Librerías cargadas")

Librerías cargadas


### Funciones de limpieza y preparación

Reutilizamos las funciones del Laboratorio 1.

In [2]:
SIN_TILDES = str.maketrans('áàäâéèëêíìïîóòöôúùüûñ', 'aaaaeeeeiiiioooouuuun')

def normalizar_texto(v):
    if pd.isna(v):
        return None
    return str(v).lower().strip().rstrip('.').translate(SIN_TILDES)

mapa_sectores = {
    'n':'N','norte':'N','north':'N',      'ne':'NE','noreste':'NE','northeast':'NE',
    'e':'E','este':'E','east':'E',        'se':'SE','sureste':'SE','southeast':'SE',
    's':'S','sur':'S','south':'S',        'so':'SO','suroeste':'SO','southwest':'SO',
    'o':'O','oeste':'O','west':'O',       'no':'NO','noroeste':'NO','northwest':'NO',
}

def estacion_del_anio(dia):
    if pd.isna(dia):
        return None
    if dia < 60:   return 'invierno'
    if dia < 152:  return 'primavera'
    if dia < 244:  return 'verano'
    if dia < 335:  return 'otono'
    return 'invierno'

def sector_del_viento(grados):
    if pd.isna(grados):
        return None
    if grados < 22.5:  return 'N'
    if grados < 67.5:  return 'NE'
    if grados < 112.5: return 'E'
    if grados < 157.5: return 'SE'
    if grados < 202.5: return 'S'
    if grados < 247.5: return 'SO'
    if grados < 292.5: return 'O'
    if grados < 337.5: return 'NO'
    return 'N'

In [3]:
def transformador_limpieza_columnas(df):
    
    X = df.copy()

    # 1. Recuperar año y día del año a partir de la fecha si estuviesen ausentes
    if 'fecha' in X.columns:
        fecha_dt = pd.to_datetime(X['fecha'], format='%Y-%m-%d', errors='coerce').fillna(
                   pd.to_datetime(X['fecha'], format='%d.%m.%Y', errors='coerce'))
        X['anio'] = X['anio'].fillna(fecha_dt.dt.year)
        X['dia_del_anio'] = X['dia_del_anio'].fillna(fecha_dt.dt.dayofyear)

    # 2. Derivación de la estación y del sector del viento
    X['estacion'] = X['dia_del_anio'].apply(estacion_del_anio)
    sector = X['sector_viento'].map(normalizar_texto).map(mapa_sectores)
    X['sector_viento'] = sector.fillna(X['direccion_viento'].apply(sector_del_viento))

    # 3. Presión: corregir la coma corrida y validar límites físicos
    for col in ['presion_media', 'presion_min', 'presion_max']:
        alto = X[col] > 1060
        X.loc[alto, col] = X.loc[alto, col] / 10
        X.loc[(X[col] < 900) | (X[col] > 1060), col] = np.nan

    # 4. Humedad: unificar escala de fracción a porcentaje y validar [0, 100]
    for col in ['humedad_media', 'humedad_min', 'humedad_max']:
        fraccion = (X[col] >= 0) & (X[col] <= 1)
        X.loc[fraccion, col] = X.loc[fraccion, col] * 100
        X.loc[(X[col] < 0) | (X[col] > 100), col] = np.nan

    # 5. Ráfagas: la rapidez nunca es negativa
    for col in ['rafaga_media', 'rafaga_min', 'rafaga_max', 'rafaga_desv']:
        X.loc[X[col] < 0, col] = np.nan

    # 6. Un día completo tiene como máximo 144 mediciones de 10 minutos
    X.loc[(X['registros_del_dia'] < 0) | (X['registros_del_dia'] > 144), 'registros_del_dia'] = np.nan

    # 7. El viento mínimo del día no puede superar al máximo
    incoherente = X['viento_min'] > X['viento_max']
    X.loc[incoherente, ['viento_min', 'viento_max']] = np.nan

    # 8. Valores extremos en la componente norte del viento
    X.loc[(X['viento_norte'] < -100) | (X['viento_norte'] > 100), 'viento_norte'] = np.nan

    return X


### Filtrado de registros y partición de los datos
Reutilizamos la lógica del laboratorio pasado: 

A continuación preparamos el conjunto de datos de entrenamiento eliminando filas duplicadas exactas y aquellas donde no se tiene registro de la variable objetivo. Posteriormente separamos las variables predictoras de la variable objetivo y realizamos la partición en conjunto de entrenamiento y conjunto de prueba con una semilla de 42 (random_state=42) y un tamaño del 25% para prueba (test_size=0.25).

In [17]:
datos = pd.read_csv("./data/Datos Lab 1.csv")
datos_modelado = datos.copy()

# Recuperar el calendario para poder deduplicar por día
fecha_dt = pd.to_datetime(datos_modelado['fecha'], format='%Y-%m-%d', errors='coerce').fillna(
           pd.to_datetime(datos_modelado['fecha'], format='%d.%m.%Y', errors='coerce'))
datos_modelado['anio'] = datos_modelado['anio'].fillna(fecha_dt.dt.year)
datos_modelado['dia_del_anio'] = datos_modelado['dia_del_anio'].fillna(fecha_dt.dt.dayofyear)

# Registros sin ninguna información de calendario
datos_modelado = datos_modelado[datos_modelado['anio'].notna() & datos_modelado['dia_del_anio'].notna()]

# Unicidad: un día aparece una sola vez, conservando la fila más completa
datos_modelado['_nulos'] = datos_modelado.isna().sum(axis=1)
datos_modelado['_orden'] = np.arange(len(datos_modelado))
datos_modelado = (datos_modelado
                  .sort_values(['_nulos', '_orden'], kind='stable')
                  .drop_duplicates(subset=['anio', 'dia_del_anio'], keep='first')
                  .sort_index()
                  .drop(columns=['_nulos', '_orden']))

# La etiqueta no se puede imputar, y los valores imposibles se descartan
datos_modelado = datos_modelado[datos_modelado['temp_max_manana'].notna()]
datos_modelado = datos_modelado[datos_modelado['temp_max_manana'] <= 45]

X = datos_modelado.drop(columns=['temp_max_manana'])
y = datos_modelado['temp_max_manana']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Registros para modelado: {len(datos_modelado)}")
print(f"  Entrenamiento: {X_train.shape[0]} filas  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Prueba       : {X_test.shape[0]} filas  ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nVariable objetivo: media {y.mean():.2f} °C, desviación {y.std():.2f} °C")

Registros para modelado: 2455
  Entrenamiento: 1841 filas  (75%)
  Prueba       : 614 filas  (25%)

Variable objetivo: media 13.78 °C, desviación 9.05 °C


### Conjunto de variables

Usamos el mismo conjunto del Modelo 1 del Laboratorio 1, que fue el de mejor desempeño. Son nueve variables seleccionadas por su aporte físico al balance térmico diario: presion_media, presion_desv, humedad_media, viento_norte, viento_este, viento_media, rafaga_max, dia_sin, dia_cos. 

La codificación cíclica: dia_del_anio es una variable circular: el día 366 es vecino del día 1, pero numéricamente son los valores más alejados. Se representa por su seno y coseno 

Esto se hizo en el lab pasado también y acá importa porque no queremos elevar al cuadrado un número tan alto, entonces se deja como un seno acotado entre 0 a 1, ya nos evitamos que la variable de calendario domine todos los términos polinomiales. 

In [18]:
VARIABLES = [
    'presion_media', 'presion_desv',
    'humedad_media',
    'viento_norte', 'viento_este',
    'viento_media', 'rafaga_max',
    'dia_sin', 'dia_cos',
]

def preparar_variables(df):
    """Limpia las columnas, construye las variables cíclicas y selecciona el conjunto final."""
    X_limpio = transformador_limpieza_columnas(df)

    dia = X_limpio['dia_del_anio']
    X_limpio['dia_sin'] = np.sin(2 * np.pi * dia / 365.25)
    X_limpio['dia_cos'] = np.cos(2 * np.pi * dia / 365.25)

    return X_limpio[VARIABLES]

# Verificación sobre el conjunto de entrenamiento
vista_previa = preparar_variables(X_train)
print(f"Variables del modelo: {len(VARIABLES)}")
display(vista_previa.describe().T[['min', 'max']].round(3))
print(f"Valores nulos por imputar: {vista_previa.isna().sum().sum()}")

Variables del modelo: 9


,min,max
presion_media,948.996,1010.286
presion_desv,0.227,12.398
humedad_media,0.466,100.000
viento_norte,-6.798,3.915
viento_este,-3.718,4.714
viento_media,0.000,24.429
rafaga_max,0.000,23.500
dia_sin,-1.000,1.000
dia_cos,-1.000,1.000


Valores nulos por imputar: 390


---
# 1. Construcción de un modelo de regresión polinomial




Seguimos la estructura de la práctica del curso: 

1. preparar_variables aplica la limpieza, construye las variables cíclicas y selecciona las nueve, luego va a dentro del pipeline para que las mismas reglas se apliquen automáticamente a cualquier dato nuevo.
2. SimpleImputer(strategy='median') rellena los nulos y se pone antes del polinomio porque PolynomialFeatures no admite NaN. Usamos la mediana por ser la más adecuada frente a los outliers que observamos.
3. Escalador  lleva las variables a una escala comparable. Es un hiperparámetro.
4. PolynomialFeatures genera potencias y productos cruzados. Va después del escalado: si multiplicáramos presion_media (≈990) por viento_media (≈3) sin escalar el producto tendría una magnitud exagerada frente a los demás términos.
5. LinearRegression es el estimador.

Usamos include_bias=False porque LinearRegression ya incorpora su propio intercepto entonces dejar ambos crearía una columna constante redundante.

Que todo esto viva dentro de un Pipeline es lo que garantiza que, en cada partición de la validación cruzada, el imputador y el escalador se reajusten solo con los datos de entrenamiento de esa partición.

In [19]:
pipeline_polinomial = Pipeline([
    ('preparacion', FunctionTransformer(preparar_variables, validate=False)),
    ('imputador',   SimpleImputer(strategy='median')),
    ('escalador',   StandardScaler()),                                  
    ('polinomio',   PolynomialFeatures(degree=2, include_bias=False)),  
    ('modelo',      LinearRegression()),
])

pipeline_polinomial

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preparacion', ...), ('imputador', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function pre...0020B0F0F6F20>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDic

Para evaluar cómo influye la complejidad en el desempeño, buscamos el grado del polinomio y etrategia de escalamiento mediante GridSearchCV con validación cruzada. Exploramos hasta grado 4 con nueve variables. el grado 5 generaría 2001 características para las 1841 muestras de entrenamiento. El grado uno se incluye como línea base para verificar si la complejidad polinomial aporta algo sobre el modelo lineal de lab 1. 


**Configuración:**

- cv=KFold(5, shuffle=True, random_state=42) para que las particiones no queden en bloques temporales contiguos y el resultado pueda ser reproducible.
- Usamos las tres métricas que pide el enunciado con refit='rmse' y return_train_score=True para poder comparar el error de entrenamiento con el de validación que es lo que nos revela el sobreajuste. 


In [20]:
espacio_busqueda = {
    'polinomio__degree': [1, 2, 3, 4],
    'escalador': [StandardScaler(), MinMaxScaler(), RobustScaler()],
}

busqueda = GridSearchCV(
    estimator=pipeline_polinomial,
    param_grid=espacio_busqueda,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring={'rmse': 'neg_root_mean_squared_error',
             'mae':  'neg_mean_absolute_error',
             'r2':   'r2'},
    refit='rmse',
    return_train_score=True,
    n_jobs=-1,
)
busqueda.fit(X_train, y_train)
print("Búsqueda finalizada")
print("Mejor configuración:", busqueda.best_params_)
print(f"Mejor RMSE en validación cruzada: {-busqueda.best_score_:.4f} °C")

Búsqueda finalizada
Mejor configuración: {'escalador': StandardScaler(), 'polinomio__degree': 2}
Mejor RMSE en validación cruzada: 4.1507 °C


In [21]:
resultados = pd.DataFrame(busqueda.cv_results_)

tabla = pd.DataFrame({
    'grado':      [p['polinomio__degree'] for p in resultados['params']],
    'escalador':  [type(p['escalador']).__name__ for p in resultados['params']],
    'RMSE_train': -resultados['mean_train_rmse'],
    'RMSE_cv':    -resultados['mean_test_rmse'],
    'RMSE_std':    resultados['std_test_rmse'],
    'MAE_cv':     -resultados['mean_test_mae'],
    'R2_cv':       resultados['mean_test_r2'],
}).sort_values('RMSE_cv').reset_index(drop=True)

display(tabla.round(4))

,grado,escalador,RMSE_train,RMSE_cv,RMSE_std,MAE_cv,R2_cv
0,2,StandardScaler,4.0220,4.1507,0.2749,3.1788,0.7865
1,2,MinMaxScaler,4.0220,4.1507,0.2749,3.1788,0.7865
2,2,RobustScaler,4.0220,4.1507,0.2749,3.1788,0.7865
3,1,StandardScaler,4.4842,4.5134,0.1906,3.4713,0.7477
4,1,RobustScaler,4.4842,4.5134,0.1906,3.4713,0.7477
5,1,MinMaxScaler,4.4842,4.5134,0.1906,3.4713,0.7477
6,3,RobustScaler,3.6808,5.0980,1.3284,3.4054,0.6538
7,3,StandardScaler,3.6808,5.0980,1.3284,3.4054,0.6538
8,3,MinMaxScaler,3.6808,5.0980,1.3284,3.4054,0.6538
9,4,MinMaxScaler,2.9035,32.0959,34.7959,7.8365,-27.6921


La búsqueda selecciona grado 2 con standardscaler, con RMSE de 4.15 grados en validación cruzada frente a 4.51 del grado 1. La diferencia entre estos presenta la mejor que aporta la expansión polinomial. El error de entrenamiento baja siempre (4.48 a 4.02 a 3.68 a 2.90), pero el de validación baja y luego sube (4.51 a 4.15 a 5.10 a 32.10), y la brecha entre ambos crece. La desviación estándar entre particiones apunta a lo mismo 0.19 en grado 1, 0.27 en grado 2, y luego se dispara a 1.33 y 34.80 en los grados 3 y 4. En grado 4 el R^2 en validación es −27.69, es decir, el modelo predice mucho peor que la media constante. También se observa que el escalador no cambia el resultado para ningún grado. 

Ahora vamos a repartir las tres métricas que pide el enunciado sobre los tres conjuntos:

In [22]:
mejor_modelo_polinomial = busqueda.best_estimator_

def calcular_metricas(modelo, X_datos, y_datos):
    predicciones = modelo.predict(X_datos)
    return {'RMSE': np.sqrt(mean_squared_error(y_datos, predicciones)),
            'MAE':  mean_absolute_error(y_datos, predicciones),
            'R2':   r2_score(y_datos, predicciones)}

fila_ganadora = tabla.iloc[0]
resumen = pd.DataFrame([
    {'Conjunto': 'Entrenamiento', **{k: round(v, 4)
     for k, v in calcular_metricas(mejor_modelo_polinomial, X_train, y_train).items()}},
    {'Conjunto': 'Validación cruzada', 'RMSE': round(fila_ganadora['RMSE_cv'], 4),
     'MAE': round(fila_ganadora['MAE_cv'], 4), 'R2': round(fila_ganadora['R2_cv'], 4)},
    {'Conjunto': 'Prueba (no visto)', **{k: round(v, 4)
     for k, v in calcular_metricas(mejor_modelo_polinomial, X_test, y_test).items()}},
])
display(resumen)

print(f"Desviación estándar del RMSE entre particiones: {fila_ganadora['RMSE_std']:.4f} °C")
print(f"Número de características generadas: "
      f"{mejor_modelo_polinomial.named_steps['polinomio'].n_output_features_}")

,Conjunto,RMSE,MAE,R2
0,Entrenamiento,4.0356,3.0803,0.7997
1,Validación cruzada,4.1507,3.1788,0.7865
2,Prueba (no visto),4.2810,3.2611,0.7795


Desviación estándar del RMSE entre particiones: 0.2749 °C
Número de características generadas: 54
